# Прогнозирование аварийности на дорогах Москвы

Автор: Носов Данила
Дата: 6 июня 2026 
Источник данных: dtp-stat.ru (открытые данные ГИБДД)

## Описание проекта

Задача - предсказать количество дорожно-транспортных происшествий в ячейках пространственной сетки Москвы (шаг 0.005°, ~550 м) на основе:
погодных условий
освещения
состояния дорожного покрытия
координат

Целевая метрика: количество ДТП в ячейке (`crash_count`).

0. Импорт библиотек

In [ ]:
import folium
import pandas as pd
import ast
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

2. Функции для парсинга и вспомогательные

In [ ]:
def parse_point(lat_long_str):
    lat_long_dict = ast.literal_eval(lat_long_str)
    return pd.Series([lat_long_dict.get('lat'), lat_long_dict.get('long')])

def get_color(crash_num):
    if crash_num <= 2:
        return 'green'
    elif crash_num <= 8:
        return 'lightgreen'
    elif crash_num <= 19:
        return 'yellow'
    elif crash_num <= 33:
        return 'orange'
    else:
        return 'red'

def parse_road_conditions(cond_str):
    if pd.isna(cond_str):
        return []
    try:
        return ast.literal_eval(cond_str)
    except:
        return []

def get_road_category(conditions_list):
    if not isinstance(conditions_list, list):
        return 'other'
    full_text = ' '.join(conditions_list)
    if any(word in full_text for word in ['Дефекты покрытия', 'Неровное покрытие', 'Низкие сцепные качества покрытия', 'Недостатки зимнего содержания']):
        return 'road_surface'
    elif any(word in full_text for word in ['Отсутствие, плохая различимость горизонтальной разметки проезжей части', 'Отсутствие, плохая различимость вертикальной разметки', 'Отсутствие дорожных знаков в необходимых местах', 'Неправильное применение, плохая видимость дорожных знаков']):
        return 'markings_signs'
    elif any(word in full_text for word in ['Неисправность светофора', 'Плохая видимость светофора', 'Отсутствие освещения', 'Недостаточное освещение', 'Неисправное освещение', 'Плохая видимость световозвращателей, размещенных на дорожных ограждениях']):
        return 'lights'
    elif any(word in full_text for word in ['Отсутствие пешеходных ограждений в необходимых местах', 'Отсутствие дорожных ограждений в необходимых местах', 'Несоответствие дорожных ограждений предъявляемым требованиям']):
        return 'barriers'
    elif any(word in full_text for word in ['Отсутствие тротуаров (пешеходных дорожек)', 'Отсутствие элементов обустройства остановочного пункта общественного пассажирского транспорта', 'Отсутствие направляющих устройств и световозвращающих элементов на них', 'Неудовлетворительное состояние обочин', 'Неудовлетворительное состояние разделительной полосы', 'Несоответствие люков смотровых колодцев и ливневой канализации предъявляемым требованиям']):
        return 'infrastructure'
    elif any(word in full_text for word in ['Отсутствие временных ТСОД в местах проведения работ', 'Сужение проезжей части, наличие препятствий, затрудняющих движение транспортных средств', 'Ограничение видимости', 'Нарушения в размещении наружной рекламы', 'Несоответствие железнодорожного переезда предъявляемым требованиям', 'Иные недостатки', 'Не установлены']):
        return 'temporary_other'
    return 'other'

def parse_weather(weather_str):
    if pd.isna(weather_str):
        return []
    try:
        return ast.literal_eval(weather_str)
    except:
        return []

def get_weather_category(weather_list):
    if not isinstance(weather_list, list):
        return 'other'
    full_text = ' '.join(weather_list)
    if 'Ясно' in full_text:
        return 'Ясно'
    elif 'Пасмурно' in full_text:
        return 'Пасмурно'
    elif 'Дождь' in full_text:
        return 'Дождь'
    elif 'Снегопад' in full_text:
        return 'Снегопад'
    elif 'Туман' in full_text:
        return 'Туман'
    elif 'Метель' in full_text:
        return 'Метель'
    elif 'Температура выше +30С' in full_text:
        return 'Температура выше +30С'
    return 'other'

3. Конфигурация и загрузка данных
Загрузка и очистка данных

Удалены служебные колонки (`id`, `region`, `vehicles` и др.)
Из колонки `point` извлечены координаты `lat` и `lon`
Отфильтрованы ДТП с нулевыми координатами (0,0) и за пределами Москвы

In [ ]:
GRID_SIZE = 0.005
LAT_MIN, LAT_MAX = 55.4, 56.1
LON_MIN, LON_MAX = 37.1, 37.9

df = pd.read_csv('moskva_data.csv')

cols_to_drop = ['id', 'tags', 'nearby', 'region', 'scheme', 'address', 'category', 'severity',
                'dead_count', 'gibdd_number', 'participants', 'injured_count', 'parent_region',
                'participants_count', 'geometry', 'participant_categories', 'vehicles']

cols_to_drop_existing = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=cols_to_drop_existing)

4. Парсинг координат и фильтрация по Москве

In [ ]:
df[['lat', 'long']] = df['point'].apply(parse_point)
df = df[~((df['lat'] == 0.0) & (df['long'] == 0.0))]
df = df.dropna(subset=['lat', 'long'])
df = df[
    (df['lat'].between(LAT_MIN, LAT_MAX)) &
    (df['long'].between(LON_MIN, LON_MAX))
]

5. Создание карты и отрисовка полигонов (визуализация реальных ДТП)

Карта Москвы разбита на квадраты 550×550 м.  
Цвет квадрата показывает количество ДТП:

зелёный ≤ 2
жёлтый 9–19
красный ≥ 34

Карта интерактивна: при клике на квадрат отображается его ID и число ДТП.

In [ ]:
moscow_map = folium.Map(location=[df['lat'][0], df['long'][0]], zoom_start=11)

df['cell_lat'] = (df['lat'] / GRID_SIZE).astype(int)
df['cell_long'] = (df['long'] / GRID_SIZE).astype(int)
df['cell_id'] = df['cell_lat'].astype(str) + '_' + df['cell_long'].astype(str)

crash_count = df.groupby('cell_id').size().reset_index(name='crash_count')

for index, row in crash_count.iterrows():
    cell_id = row['cell_id']
    crash_num = row['crash_count']
    cell_lat_min, cell_lon_min = map(int, cell_id.split('_'))
    cell_lat_max = (cell_lat_min + 1) * GRID_SIZE
    cell_lat_min = cell_lat_min * GRID_SIZE
    cell_lon_max = (cell_lon_min + 1) * GRID_SIZE
    cell_lon_min = cell_lon_min * GRID_SIZE
    
    locations = [
        [cell_lat_min, cell_lon_min],
        [cell_lat_max, cell_lon_min],
        [cell_lat_max, cell_lon_max],
        [cell_lat_min, cell_lon_max]
    ]
    
    folium.Polygon(
        locations=locations,
        color='black',
        weight=1,
        fill_color=get_color(crash_num),
        fill_opacity=0.1,
        popup=f"Ячейка: {cell_id}<br>ДТП: {crash_num}"
    ).add_to(moscow_map)

6. Парсинг дорожных условий и погоды
`road_conditions` — 6 категорий (покрытие, разметка, светофоры, ограждения, инфраструктура, прочее)
`weather` — 7 категорий (Ясно, Пасмурно, Дождь, Снегопад, Туман, Метель, Жара)
Временные признаки: час, день недели, месяц

In [ ]:
df['road_conditions_parsed'] = df['road_conditions'].apply(parse_road_conditions)
df['road_category'] = df['road_conditions_parsed'].apply(get_road_category)

df['weather_parsed'] = df['weather'].apply(parse_weather)
df['weather_category'] = df['weather_parsed'].apply(get_weather_category)

df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month

df['cell_lat'] = (df['lat'] / GRID_SIZE).astype(int)
df['cell_long'] = (df['long'] / GRID_SIZE).astype(int)

7. One-hot encoding и подготовка признаков

In [ ]:
light_dummies = pd.get_dummies(df['light'], prefix='light').astype(int)
weather_dummies = pd.get_dummies(df['weather_category'], prefix='weather').astype(int)
road_dummies = pd.get_dummies(df['road_category'], prefix='road').astype(int)
df = pd.concat([df, light_dummies, weather_dummies, road_dummies], axis=1)

df['cell_lat'] = (df['lat'] / GRID_SIZE).astype(int)
df['cell_long'] = (df['long'] / GRID_SIZE).astype(int)

binary_cols = [col for col in df.columns if col.startswith(('light_', 'weather_', 'road_'))]
coord_cols = ['cell_lat', 'cell_long']

cell_features = df.groupby('cell_id').agg(
    {**{col: 'sum' for col in binary_cols},
     **{col: 'first' for col in coord_cols}}
).reset_index()

cell_features = cell_features.merge(crash_count, on='cell_id', how='left')
cell_features = cell_features.loc[:, ~cell_features.columns.duplicated()]

8. Преобразование в доли (устранение data leakage)
Было обнаружено, что сумма бинарных признаков равна `crash_count`.  
Если оставить суммы, модель просто выучит это равенство.

Решение: перевести суммы в **доли** — каждый признак делится на общее число ДТП в ячейке.

In [ ]:
light_cols = [c for c in cell_features.columns if c.startswith('light_')]
weather_cols = [c for c in cell_features.columns if c.startswith('weather_')]
road_cols = [c for c in cell_features.columns if c.startswith('road_')]

for col in light_cols + weather_cols + road_cols:
    if isinstance(cell_features[col].iloc[0], list):
        cell_features[col] = cell_features[col].apply(lambda x: x[0] if len(x) > 0 else 0)
    cell_features[col] = pd.to_numeric(cell_features[col], errors='coerce').fillna(0)

cell_features['lat_center'] = (cell_features['cell_lat'] + 0.5) * GRID_SIZE
cell_features['lon_center'] = (cell_features['cell_long'] + 0.5) * GRID_SIZE
cell_features = cell_features.drop(columns=['cell_lat', 'cell_long'])

light_cols = [c for c in cell_features.columns if c.startswith('light_')]
weather_cols = [c for c in cell_features.columns if c.startswith('weather_')]
road_cols = [c for c in cell_features.columns if c.startswith('road_')]

cell_features['crash_count_safe'] = cell_features['crash_count'].replace(0, 1)

for col in light_cols + weather_cols + road_cols:
    cell_features[col] = cell_features[col] / cell_features['crash_count_safe']

cell_features = cell_features.drop(columns=['crash_count_safe'])

9. Подготовка X, y и обучение модели
Алгоритм: Random Forest Regressor (100 деревьев)
Разделение данных: 80% — обучение, 20% — тест
Цель `crash_count`

In [ ]:
X = cell_features.drop(columns=[
    'cell_id', 'crash_count',
    'road_conditions', 'road_conditions_parsed', 'road_category',
    'weather_parsed', 'weather_category'
], errors='ignore')
y = cell_features['crash_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)

model = RandomForestRegressor(n_estimators=100, n_jobs=4, random_state=5)
model.fit(X_train, y_train)

prediction = model.predict(X_test)

print(f'MAE: {mean_absolute_error(y_test, prediction):.2f}')
print(f'R²: {r2_score(y_test, prediction):.3f}')

10. Важность признаков и график

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

top10 = importances.head(10)
plt.figure(figsize=(10, 6))
plt.barh(top10['feature'], top10['importance'])
plt.xlabel('Важность')
plt.title('10 самых значимых факторов, влияющих на количество ДТП')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

11. График реальные vs предсказанные

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, prediction, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Реальные ДТП')
plt.ylabel('Предсказанные ДТП')
plt.title('Реальные против предсказанных значений')
plt.tight_layout()
plt.savefig('real_vs_pred.png')
plt.show()

12. Добавление предсказаний на карту и сохранение

In [ ]:
for (lat, lon), pred in zip(X_test[['lat_center', 'lon_center']].values, prediction):
    folium.Marker(
        location=[lat, lon],
        popup=f'Предсказанное количество ДТП: {pred:.1f}'
    ).add_to(moscow_map)

moscow_map.save('moscow_heatmap.html')